In [ ]:
# mobilenetv4 specific training code

# set up, local copy
!pip install -q timm albumentations scikit-learn seaborn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import timm
from pathlib import Path
from tqdm import tqdm
import numpy as np
import random
import warnings
import zipfile
import time
import copy
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns
import matplotlib.pyplot as plt

cv2.setNumThreads(0)
warnings.filterwarnings("ignore")

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount('/content/drive')

# path set up (please change to correct path if this default is incorrect)
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/dataset_resplit.zip")
LOCAL_ZIP_PATH = Path("/content/dataset_resplit.zip")
LOCAL_DATA_ROOT = Path("/content/dataset_resplit")

if not LOCAL_ZIP_PATH.exists():
    print("Copying dataset_resplit.zip from Drive to local disk...")
    t0 = time.time()
    !cp "{DRIVE_ZIP_PATH}" "{LOCAL_ZIP_PATH}"
    print(f"Copy done in {time.time() - t0:.1f}s")
else:
    print("Local zip already present, skipping copy.")

if not LOCAL_DATA_ROOT.exists():
    print("Extracting dataset locally...")
    t0 = time.time()
    with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA_ROOT)
    print(f"Extraction done in {time.time() - t0:.1f}s")
else:
    print("Local dataset folder already present, skipping extraction.")

DATA_ROOT = LOCAL_DATA_ROOT / "dataset"
assert (DATA_ROOT / "Train").exists(), (
    f"Expected Train/ under {DATA_ROOT} but it's missing, check the zip structure with "
    f"'!unzip -l {LOCAL_ZIP_PATH} | head -30' and adjust DATA_ROOT manually."
)
print(f"\nDATA_ROOT confirmed: {DATA_ROOT}")

# pretrained checkpoint, normalization
MODEL_NAME = 'mobilenetv4_conv_aa_large.e230_r384_in12k_ft_in1k'
try:
    _cfg = timm.get_pretrained_cfg(MODEL_NAME)
except Exception as e:
    print(f"Could not resolve config for {MODEL_NAME} ({e}), falling back to mobilenetv4_conv_aa_large.e600_r384_in1k.")
    MODEL_NAME = 'mobilenetv4_conv_aa_large.e600_r384_in1k'
    _cfg = timm.get_pretrained_cfg(MODEL_NAME)

NORM_MEAN = list(_cfg.mean)
NORM_STD = list(_cfg.std)
print(f"Resolved checkpoint: {MODEL_NAME}")
print(f"Using normalization -> mean={NORM_MEAN}, std={NORM_STD}")


# transforms
IMG_SIZE = 384

train_transform = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0), ratio=(0.85, 1.15), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.15),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=0.5),
    A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

val_test_transform = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])


class MushroomDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_names = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}

        for class_name in self.class_names:
            class_dir = self.root_dir / class_name
            for img_file in class_dir.glob("*.jpg"):
                self.image_paths.append(img_file)
                self.labels.append(self.class_to_idx[class_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(str(img_path))

        if image is None:  # corrupted image
            print(f"Corrupted image skipped (resampling): {img_path.name}")
            fallback_idx = random.randint(0, len(self.image_paths) - 1)
            return self.__getitem__(fallback_idx)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, label


# dataloaders
batch_size = 64

train_dataset = MushroomDataset(DATA_ROOT / "Train", transform=train_transform)
val_dataset   = MushroomDataset(DATA_ROOT / "Val",   transform=val_test_transform)
test_dataset  = MushroomDataset(DATA_ROOT / "Test",  transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                           num_workers=8, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                           num_workers=8, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                           num_workers=8, pin_memory=True, persistent_workers=True)

print(f"Batch size: {batch_size} | Resolution: {IMG_SIZE}x{IMG_SIZE}")
print(f"Train/Val/Test sizes: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")
print(f"Classes: {train_dataset.class_names}")


# model, no freezing full parameters.
model = timm.create_model(MODEL_NAME, pretrained=True,
                           num_classes=len(train_dataset.class_names),
                           drop_path_rate=0.2)
model.to(device)
print(f"Loaded checkpoint: {MODEL_NAME}")

for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")


# layer-wise learning rate decay

def get_mnv4_layer_id(param_name, num_blocks=5):
    if param_name.startswith("conv_stem") or param_name.startswith("bn1"):
        return 0
    if param_name.startswith("blocks"):
        block_num = int(param_name.split(".")[1])
        return block_num + 1
    # conv_head, norm_head, classifier, or anything else not matched above
    return num_blocks + 2

num_layers = 5 + 2  # stem(0) + 5 blocks(1-5) + head group(7)
base_lr = 5e-5
layer_decay = 0.85

param_groups = {}
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    layer_id = get_mnv4_layer_id(name)
    lr_scale = layer_decay ** (num_layers - layer_id)
    group_key = layer_id
    if group_key not in param_groups:
        param_groups[group_key] = {"params": [], "lr": base_lr * lr_scale}
    param_groups[group_key]["params"].append(param)

print("\nLayer-wise LR schedule:")
for layer_id in sorted(param_groups.keys()):
    n_params = sum(p.numel() for p in param_groups[layer_id]["params"])
    print(f"  Layer {layer_id}: LR={param_groups[layer_id]['lr']:.2e}, params={n_params:,}")

optimizer = torch.optim.AdamW(list(param_groups.values()), weight_decay=0.05)


# loss, schedule, ema, mixup/cutmix
class_weights = compute_class_weight(
    'balanced', classes=np.unique(train_dataset.labels), y=train_dataset.labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"\nClass weights: {dict(zip(train_dataset.class_names, class_weights.cpu().numpy().round(3)))}")

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# smoke test
SMOKE_TEST = False
num_epochs = 5 if SMOKE_TEST else 15
early_stop_patience = 3 if SMOKE_TEST else 5
warmup_epochs = 1 if SMOKE_TEST else 2

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, total_iters=warmup_epochs
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-7
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
)

steps_per_epoch = max(1, len(train_dataset) // batch_size)
target_window_steps = int(steps_per_epoch * 1.5)
ema_decay = 1 - (1 / target_window_steps)
print(f"Steps/epoch: {steps_per_epoch} | EMA decay: {ema_decay:.5f} (window ~{target_window_steps} steps)")

ema_model = copy.deepcopy(model)
for p in ema_model.parameters():
    p.requires_grad = False

def update_ema(ema_model, model, decay):
    with torch.no_grad():
        ema_params = dict(ema_model.named_parameters())
        for name, param in model.named_parameters():
            ema_params[name].mul_(decay).add_(param.detach(), alpha=1 - decay)
        ema_buffers = dict(ema_model.named_buffers())
        for name, buf in model.named_buffers():
            ema_buffers[name].copy_(buf)

mixup_alpha = 0.2
cutmix_alpha = 1.0
mixup_cutmix_prob = 0.5

def rand_bbox(size, lam):
    H, W = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    x2, y2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2

def apply_mixup_cutmix(images, labels, num_classes):
    if random.random() > mixup_cutmix_prob:
        return images, labels, labels, 1.0

    batch_size = images.size(0)
    perm = torch.randperm(batch_size, device=images.device)
    labels_b = labels[perm]

    if random.random() < 0.5:  # mixup
        lam = np.random.beta(mixup_alpha, mixup_alpha)
        images = lam * images + (1 - lam) * images[perm]
    else:  # cutmix
        lam = np.random.beta(cutmix_alpha, cutmix_alpha)
        x1, y1, x2, y2 = rand_bbox(images.size(), lam)
        images[:, :, y1:y2, x1:x2] = images[perm][:, :, y1:y2, x1:x2]
        lam = 1 - ((x2 - x1) * (y2 - y1) / (images.size(-1) * images.size(-2)))

    return images, labels, labels_b, lam

def mixup_cutmix_loss(criterion, outputs, labels_a, labels_b, lam):
    return lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)


# model save dir
MNV4_ROOT = Path("/content/drive/MyDrive/Model/MobileNetV4")
MNV4_ROOT.mkdir(parents=True, exist_ok=True)

existing_runs = [
    int(d.name.replace("MobileNetV4_Training", ""))
    for d in MNV4_ROOT.iterdir()
    if d.is_dir() and d.name.startswith("MobileNetV4_Training")
    and d.name.replace("MobileNetV4_Training", "").isdigit()
]
next_run_number = max(existing_runs, default=0) + 1

MODEL_SAVE_DIR = MNV4_ROOT / f"MobileNetV4_Training{next_run_number}"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving this run's outputs to: {MODEL_SAVE_DIR}")


# training
best_val_acc = 0.0
best_epoch = 0
best_source = None
epochs_no_improve = 0
grad_clip_norm = 1.0

print("\nStarting training...\n")


def evaluate_model(m, loader, desc=""):
    m.eval()
    correct, total, loss_sum = 0, 0, 0.0
    preds_out, labels_out = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc=desc, leave=False):
            images, labels = images.to(device), labels.to(device)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                outputs = m(images)
                loss = criterion(outputs, labels)
            loss_sum += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            preds_out.extend(predicted.cpu().numpy())
            labels_out.extend(labels.cpu().numpy())
    acc = 100 * correct / total if total > 0 else 0.0
    avg_loss = loss_sum / len(loader) if len(loader) > 0 else 0.0
    return acc, avg_loss, preds_out, labels_out


for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} [Train]"):
        images, labels = images.to(device), labels.to(device)

        images, labels_a, labels_b, lam = apply_mixup_cutmix(images, labels, len(train_dataset.class_names))

        optimizer.zero_grad()

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            outputs = model(images)
            loss = mixup_cutmix_loss(criterion, outputs, labels_a, labels_b, lam)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()
        update_ema(ema_model, model, ema_decay)

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels_a).sum().item()

    train_acc = 100 * train_correct / train_total if train_total > 0 else 0.0

    raw_val_acc, raw_val_loss, raw_preds, raw_labels = evaluate_model(model, val_loader, desc=f"Epoch {epoch}/{num_epochs} [Val-RAW]")
    ema_val_acc, ema_val_loss, ema_preds, ema_labels = evaluate_model(ema_model, val_loader, desc=f"Epoch {epoch}/{num_epochs} [Val-EMA]")

    if ema_val_acc >= raw_val_acc:
        val_acc, val_loss, all_preds, all_labels, source = ema_val_acc, ema_val_loss, ema_preds, ema_labels, "EMA"
        winning_state = ema_model.state_dict()
    else:
        val_acc, val_loss, all_preds, all_labels, source = raw_val_acc, raw_val_loss, raw_preds, raw_labels, "RAW"
        winning_state = model.state_dict()

    scheduler.step()

    current_lrs = [g['lr'] for g in optimizer.param_groups]
    print(f"Epoch {epoch} | Train Acc (approx): {train_acc:.2f}% | Val Acc: {val_acc:.2f}% ({source}) "
          f"[raw={raw_val_acc:.2f}% ema={ema_val_acc:.2f}%] | LR range: [{min(current_lrs):.2e}, {max(current_lrs):.2e}]")

    if epoch % 3 == 0 or epoch == num_epochs:
        torch.save(winning_state, MODEL_SAVE_DIR / f"mobilenetv4_epoch_{epoch}.pth")

        report = classification_report(all_labels, all_preds, target_names=train_dataset.class_names, digits=4)
        print(f"\n--- Epoch {epoch} Evaluation ({source} weights) ---")
        print(report)

        cm = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=train_dataset.class_names,
                    yticklabels=train_dataset.class_names)
        plt.title(f'Confusion Matrix - Epoch {epoch} ({source})')
        plt.savefig(MODEL_SAVE_DIR / f"confusion_matrix_epoch_{epoch}.png")
        plt.close()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        best_source = source
        epochs_no_improve = 0
        torch.save(winning_state, MODEL_SAVE_DIR / "best_mobilenetv4_model.pth")
        print(f"   Best model updated at epoch {epoch}! (Val Acc: {val_acc:.2f}%, source: {source})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stop_patience:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {early_stop_patience} epochs).")
            break


# final test - with tta
print("\n" + "="*70)
print("FINAL TEST EVALUATION (with TTA)...")
print(f"Loading best checkpoint from epoch {best_epoch} (source: {best_source}, Val Acc: {best_val_acc:.2f}%)")

model.load_state_dict(torch.load(MODEL_SAVE_DIR / "best_mobilenetv4_model.pth"))
model.to(device)
model.eval()


@torch.no_grad()
def predict_tta(m, images, dev):
    images = images.to(dev)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits1 = m(images)
        logits2 = m(torch.flip(images, dims=[3]))
    return (logits1.softmax(1) + logits2.softmax(1)) / 2


all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Final Test (TTA)"):
        labels = labels.to(device)
        probs = predict_tta(model, images, device)
        _, predicted = torch.max(probs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = 100 * sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
report = classification_report(all_labels, all_preds, target_names=test_dataset.class_names, digits=4)

print(f"\nFINAL TEST ACCURACY (TTA): {test_acc:.2f}% (Best Val at epoch {best_epoch}, Val Acc: {best_val_acc:.2f}%, source: {best_source})")
print(report)

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_dataset.class_names,
            yticklabels=test_dataset.class_names)
plt.title('Final Test Confusion Matrix (TTA)')
plt.savefig(MODEL_SAVE_DIR / "confusion_matrix_final_test_tta.png")
plt.close()

print(f"\nAll models saved in: {MODEL_SAVE_DIR}")


In [ ]:
# swin specific training code

# set up, local copy
!pip install -q timm albumentations scikit-learn seaborn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import timm
from pathlib import Path
from tqdm import tqdm
import numpy as np
import random
import warnings
import zipfile
import time
import copy
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns
import matplotlib.pyplot as plt

cv2.setNumThreads(0)
warnings.filterwarnings("ignore")

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount('/content/drive')

# path set up (please change to correct path if this default is incorrect)
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/dataset_resplit.zip")
LOCAL_ZIP_PATH = Path("/content/dataset_resplit.zip")
LOCAL_DATA_ROOT = Path("/content/dataset_resplit")

if not LOCAL_ZIP_PATH.exists():
    print("Copying dataset_resplit.zip from Drive to local disk...")
    t0 = time.time()
    !cp "{DRIVE_ZIP_PATH}" "{LOCAL_ZIP_PATH}"
    print(f"Copy done in {time.time() - t0:.1f}s")
else:
    print("Local zip already present, skipping copy.")

if not LOCAL_DATA_ROOT.exists():
    print("Extracting dataset locally...")
    t0 = time.time()
    with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA_ROOT)
    print(f"Extraction done in {time.time() - t0:.1f}s")
else:
    print("Local dataset folder already present, skipping extraction.")

DATA_ROOT = LOCAL_DATA_ROOT / "dataset"
assert (DATA_ROOT / "Train").exists(), (
    f"Expected Train/ under {DATA_ROOT} but it's missing, check the zip structure with "
    f"'!unzip -l {LOCAL_ZIP_PATH} | head -30' and adjust DATA_ROOT manually."
)
print(f"\nDATA_ROOT confirmed: {DATA_ROOT}")


MODEL_NAME = 'swin_base_patch4_window12_384.ms_in22k_ft_in1k'
try:
    _cfg = timm.get_pretrained_cfg(MODEL_NAME)
except Exception as e:
    print(f"Could not resolve config for {MODEL_NAME} ({e}), falling back to swin_base_patch4_window7_224.ms_in22k_ft_in1k.")
    MODEL_NAME = 'swin_base_patch4_window7_224.ms_in22k_ft_in1k'
    _cfg = timm.get_pretrained_cfg(MODEL_NAME)

NORM_MEAN = list(_cfg.mean)
NORM_STD = list(_cfg.std)
print(f"Resolved checkpoint: {MODEL_NAME}")
print(f"Using normalization -> mean={NORM_MEAN}, std={NORM_STD}")


# transforms
IMG_SIZE = 384

train_transform = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0), ratio=(0.85, 1.15), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.15),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=0.5),
    A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

val_test_transform = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])


class MushroomDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_names = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}

        for class_name in self.class_names:
            class_dir = self.root_dir / class_name
            for img_file in class_dir.glob("*.jpg"):
                self.image_paths.append(img_file)
                self.labels.append(self.class_to_idx[class_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(str(img_path))

        if image is None:
            print(f"Corrupted image skipped (resampling): {img_path.name}")
            fallback_idx = random.randint(0, len(self.image_paths) - 1)
            return self.__getitem__(fallback_idx)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, label


# dataloaders
batch_size = 32

train_dataset = MushroomDataset(DATA_ROOT / "Train", transform=train_transform)
val_dataset   = MushroomDataset(DATA_ROOT / "Val",   transform=val_test_transform)
test_dataset  = MushroomDataset(DATA_ROOT / "Test",  transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                           num_workers=8, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                           num_workers=8, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                           num_workers=8, pin_memory=True, persistent_workers=True)

print(f"Batch size: {batch_size} | Resolution: {IMG_SIZE}x{IMG_SIZE}")
print(f"Train/Val/Test sizes: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")
print(f"Classes: {train_dataset.class_names}")


# model
model = timm.create_model(MODEL_NAME, pretrained=True,
                           num_classes=len(train_dataset.class_names),
                           drop_path_rate=0.2)
model.to(device)
print(f"Loaded checkpoint: {MODEL_NAME}")

for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")


# llrd
def get_swin_layer_id(param_name, num_stages=4):
    if param_name.startswith("patch_embed"):
        return 0
    if param_name.startswith("layers"):
        stage_num = int(param_name.split(".")[1])
        return stage_num + 1
    return num_stages + 2

num_layers = 4 + 2
base_lr = 5e-5
layer_decay = 0.85

param_groups = {}
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    layer_id = get_swin_layer_id(name)
    lr_scale = layer_decay ** (num_layers - layer_id)
    group_key = layer_id
    if group_key not in param_groups:
        param_groups[group_key] = {"params": [], "lr": base_lr * lr_scale}
    param_groups[group_key]["params"].append(param)

print("\nLayer-wise LR schedule:")
for layer_id in sorted(param_groups.keys()):
    n_params = sum(p.numel() for p in param_groups[layer_id]["params"])
    print(f"  Layer {layer_id}: LR={param_groups[layer_id]['lr']:.2e}, params={n_params:,}")

optimizer = torch.optim.AdamW(list(param_groups.values()), weight_decay=0.05)


# loss, schedule, ema, mixup/cutmix
class_weights = compute_class_weight(
    'balanced', classes=np.unique(train_dataset.labels), y=train_dataset.labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"\nClass weights: {dict(zip(train_dataset.class_names, class_weights.cpu().numpy().round(3)))}")

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# smoke test
SMOKE_TEST = False
num_epochs = 5 if SMOKE_TEST else 15
early_stop_patience = 3 if SMOKE_TEST else 5
warmup_epochs = 1 if SMOKE_TEST else 2

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, total_iters=warmup_epochs
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-7
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
)

# ema
steps_per_epoch = max(1, len(train_dataset) // batch_size)
target_window_steps = int(steps_per_epoch * 1.5)
ema_decay = 1 - (1 / target_window_steps)
print(f"Steps/epoch: {steps_per_epoch} | EMA decay: {ema_decay:.5f} (window ~{target_window_steps} steps)")

ema_model = copy.deepcopy(model)
for p in ema_model.parameters():
    p.requires_grad = False

def update_ema(ema_model, model, decay):
    with torch.no_grad():
        ema_params = dict(ema_model.named_parameters())
        for name, param in model.named_parameters():
            ema_params[name].mul_(decay).add_(param.detach(), alpha=1 - decay)
        ema_buffers = dict(ema_model.named_buffers())
        for name, buf in model.named_buffers():
            ema_buffers[name].copy_(buf)

# mixup / cutmix
mixup_alpha = 0.2
cutmix_alpha = 1.0
mixup_cutmix_prob = 0.5

def rand_bbox(size, lam):
    H, W = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    x2, y2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2

def apply_mixup_cutmix(images, labels, num_classes):
    if random.random() > mixup_cutmix_prob:
        return images, labels, labels, 1.0

    batch_size = images.size(0)
    perm = torch.randperm(batch_size, device=images.device)
    labels_b = labels[perm]

    if random.random() < 0.5:  # mixup
        lam = np.random.beta(mixup_alpha, mixup_alpha)
        images = lam * images + (1 - lam) * images[perm]
    else:  # cutmix
        lam = np.random.beta(cutmix_alpha, cutmix_alpha)
        x1, y1, x2, y2 = rand_bbox(images.size(), lam)
        images[:, :, y1:y2, x1:x2] = images[perm][:, :, y1:y2, x1:x2]
        lam = 1 - ((x2 - x1) * (y2 - y1) / (images.size(-1) * images.size(-2)))

    return images, labels, labels_b, lam

def mixup_cutmix_loss(criterion, outputs, labels_a, labels_b, lam):
    return lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)


# model save dir
SWIN_ROOT = Path("/content/drive/MyDrive/Model/Swin")
SWIN_ROOT.mkdir(parents=True, exist_ok=True)

existing_runs = [
    int(d.name.replace("Swin_Training", ""))
    for d in SWIN_ROOT.iterdir()
    if d.is_dir() and d.name.startswith("Swin_Training")
    and d.name.replace("Swin_Training", "").isdigit()
]
next_run_number = max(existing_runs, default=0) + 1

MODEL_SAVE_DIR = SWIN_ROOT / f"Swin_Training{next_run_number}"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving this run's outputs to: {MODEL_SAVE_DIR}")


# training
best_val_acc = 0.0
best_epoch = 0
best_source = None
epochs_no_improve = 0
grad_clip_norm = 1.0

print("\nStarting training...\n")


def evaluate_model(m, loader, desc=""):
    m.eval()
    correct, total, loss_sum = 0, 0, 0.0
    preds_out, labels_out = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc=desc, leave=False):
            images, labels = images.to(device), labels.to(device)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                outputs = m(images)
                loss = criterion(outputs, labels)
            loss_sum += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            preds_out.extend(predicted.cpu().numpy())
            labels_out.extend(labels.cpu().numpy())
    acc = 100 * correct / total if total > 0 else 0.0
    avg_loss = loss_sum / len(loader) if len(loader) > 0 else 0.0
    return acc, avg_loss, preds_out, labels_out


for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} [Train]"):
        images, labels = images.to(device), labels.to(device)

        images, labels_a, labels_b, lam = apply_mixup_cutmix(images, labels, len(train_dataset.class_names))

        optimizer.zero_grad()

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            outputs = model(images)
            loss = mixup_cutmix_loss(criterion, outputs, labels_a, labels_b, lam)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()
        update_ema(ema_model, model, ema_decay)

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels_a).sum().item()

    train_acc = 100 * train_correct / train_total if train_total > 0 else 0.0

    raw_val_acc, raw_val_loss, raw_preds, raw_labels = evaluate_model(model, val_loader, desc=f"Epoch {epoch}/{num_epochs} [Val-RAW]")
    ema_val_acc, ema_val_loss, ema_preds, ema_labels = evaluate_model(ema_model, val_loader, desc=f"Epoch {epoch}/{num_epochs} [Val-EMA]")

    if ema_val_acc >= raw_val_acc:
        val_acc, val_loss, all_preds, all_labels, source = ema_val_acc, ema_val_loss, ema_preds, ema_labels, "EMA"
        winning_state = ema_model.state_dict()
    else:
        val_acc, val_loss, all_preds, all_labels, source = raw_val_acc, raw_val_loss, raw_preds, raw_labels, "RAW"
        winning_state = model.state_dict()

    scheduler.step()

    current_lrs = [g['lr'] for g in optimizer.param_groups]
    print(f"Epoch {epoch} | Train Acc (approx): {train_acc:.2f}% | Val Acc: {val_acc:.2f}% ({source}) "
          f"[raw={raw_val_acc:.2f}% ema={ema_val_acc:.2f}%] | LR range: [{min(current_lrs):.2e}, {max(current_lrs):.2e}]")

    if epoch % 3 == 0 or epoch == num_epochs:
        torch.save(winning_state, MODEL_SAVE_DIR / f"swin_epoch_{epoch}.pth")

        report = classification_report(all_labels, all_preds, target_names=train_dataset.class_names, digits=4)
        print(f"\n--- Epoch {epoch} Evaluation ({source} weights) ---")
        print(report)

        cm = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=train_dataset.class_names,
                    yticklabels=train_dataset.class_names)
        plt.title(f'Confusion Matrix - Epoch {epoch} ({source})')
        plt.savefig(MODEL_SAVE_DIR / f"confusion_matrix_epoch_{epoch}.png")
        plt.close()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        best_source = source
        epochs_no_improve = 0
        torch.save(winning_state, MODEL_SAVE_DIR / "best_swin_model.pth")
        print(f"   Best model updated at epoch {epoch}! (Val Acc: {val_acc:.2f}%, source: {source})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stop_patience:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {early_stop_patience} epochs).")
            break


# final test - with tta
print("\n" + "="*70)
print("FINAL TEST EVALUATION (with TTA)...")
print(f"Loading best checkpoint from epoch {best_epoch} (source: {best_source}, Val Acc: {best_val_acc:.2f}%)")

model.load_state_dict(torch.load(MODEL_SAVE_DIR / "best_swin_model.pth"))
model.to(device)
model.eval()


@torch.no_grad()
def predict_tta(m, images, dev):
    images = images.to(dev)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits1 = m(images)
        logits2 = m(torch.flip(images, dims=[3]))
    return (logits1.softmax(1) + logits2.softmax(1)) / 2


all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Final Test (TTA)"):
        labels = labels.to(device)
        probs = predict_tta(model, images, device)
        _, predicted = torch.max(probs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = 100 * sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
report = classification_report(all_labels, all_preds, target_names=test_dataset.class_names, digits=4)

print(f"\nFINAL TEST ACCURACY (TTA): {test_acc:.2f}% (Best Val at epoch {best_epoch}, Val Acc: {best_val_acc:.2f}%, source: {best_source})")
print(report)

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_dataset.class_names,
            yticklabels=test_dataset.class_names)
plt.title('Final Test Confusion Matrix (TTA)')
plt.savefig(MODEL_SAVE_DIR / "confusion_matrix_final_test_tta.png")
plt.close()

print(f"\nAll models saved in: {MODEL_SAVE_DIR}")

In [ ]:
# manual sample (view-angle) audit

# setup

!pip install -q ipywidgets

import random
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

seed = 42
random.seed(seed)


from google.colab import drive
drive.mount('/content/drive')

LOCAL_DATA_ROOT = Path("/content/dataset_resplit")
DATA_ROOT = LOCAL_DATA_ROOT / "dataset"

SPLITS = ["Train", "Val", "Test"]
CLASS_NAMES = sorted([d.name for d in (DATA_ROOT / "Train").iterdir() if d.is_dir()])
print(f"classes found: {CLASS_NAMES}")

# 10/class x 6 classes = 60 images
N_PER_SPECIES = 10

AUDIT_CSV = Path("/content/drive/MyDrive/Model/view_angle_audit.csv")
AUDIT_CSV.parent.mkdir(parents=True, exist_ok=True)


# build sample (stratified: N_PER_SPECIES per class, across all splits)

def collect_class_images(class_name):
    paths = []
    for split in SPLITS:
        class_dir = DATA_ROOT / split / class_name
        if class_dir.exists():
            paths.extend(sorted(class_dir.glob("*.jpg")))
    return paths

if AUDIT_CSV.exists():
    print(f"found existing audit file at {AUDIT_CSV}, resuming from it.")
    audit_df = pd.read_csv(AUDIT_CSV)
else:
    print("no existing audit file found, building a new sample.")
    rows = []
    for class_name in CLASS_NAMES:
        available = collect_class_images(class_name)
        n = min(N_PER_SPECIES, len(available))
        sampled = random.sample(available, n)
        for p in sampled:
            rows.append({"image_path": str(p), "species": class_name, "viewing_angle": None})
    audit_df = pd.DataFrame(rows)
    audit_df.to_csv(AUDIT_CSV, index=False)

total_n = len(audit_df)
labeled_n = audit_df["viewing_angle"].notna().sum()
print(f"sample size: {total_n} images ({N_PER_SPECIES} per species x {len(CLASS_NAMES)} species)")
print(f"already labeled: {labeled_n}/{total_n}")


# 2. interactive labeling widget

unlabeled_idx = audit_df.index[audit_df["viewing_angle"].isna()].tolist()
progress_counter = {"done": total_n - len(unlabeled_idx)}

image_widget = widgets.Image(format='jpg', width=400)
label_widget = widgets.HTML()
progress_widget = widgets.HTML()

btn_top = widgets.Button(description="Top / Cap", button_style="info")
btn_gill = widgets.Button(description="Underside / Gill", button_style="info")
btn_side = widgets.Button(description="Side / Stem", button_style="info")
btn_mixed = widgets.Button(description="Mixed / Unclear", button_style="warning")
btn_skip = widgets.Button(description="Skip (bad image)", button_style="danger")

button_row = widgets.HBox([btn_top, btn_gill, btn_side, btn_mixed, btn_skip])
ui = widgets.VBox([progress_widget, image_widget, label_widget, button_row])


def load_next():
    if not unlabeled_idx:
        clear_output(wait=True)
        print("all images labeled! move on to the summary cell below (Click run cell again if stuck here).")
        return
    idx = unlabeled_idx[0]
    img_path = audit_df.loc[idx, "image_path"]
    species = audit_df.loc[idx, "species"]
    with open(img_path, "rb") as f:
        image_widget.value = f.read()
    label_widget.value = f"<b>species:</b> {species}<br><b>file:</b> {Path(img_path).name}"
    progress_widget.value = f"<b>{progress_counter['done']}/{total_n} labeled</b>"


def make_handler(label_value):
    def handler(_):
        idx = unlabeled_idx.pop(0)
        audit_df.loc[idx, "viewing_angle"] = label_value
        audit_df.to_csv(AUDIT_CSV, index=False)
        progress_counter["done"] += 1
        load_next()
    return handler


btn_top.on_click(make_handler("top"))
btn_gill.on_click(make_handler("gill"))
btn_side.on_click(make_handler("side"))
btn_mixed.on_click(make_handler("mixed"))
btn_skip.on_click(make_handler("skip"))

display(ui)
load_next()


# summary stats

audit_df = pd.read_csv(AUDIT_CSV)
labeled_df = audit_df[audit_df["viewing_angle"].notna() & (audit_df["viewing_angle"] != "skip")]

n_total_labeled = len(labeled_df)
n_skipped = (audit_df["viewing_angle"] == "skip").sum()
n_pending = audit_df["viewing_angle"].isna().sum()

print(f"labeled: {n_total_labeled} | skipped (bad images): {n_skipped} | still pending: {n_pending}\n")

overall_pct = (labeled_df["viewing_angle"].value_counts(normalize=True) * 100).round(1)
print("overall viewing-angle distribution:")
print(overall_pct.to_string())

print("\nper-species breakdown (%):")
per_species_pct = (
    labeled_df.groupby("species")["viewing_angle"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .unstack(fill_value=0)
)
print(per_species_pct.to_string())

top_pct = overall_pct.get("top", 0.0)
gill_pct = overall_pct.get("gill", 0.0)
side_pct = overall_pct.get("side", 0.0) + overall_pct.get("mixed", 0.0)

print("\n--- paper sentence draft ---")
print(
    f"A random sample of {n_total_labeled} images "
    f"({N_PER_SPECIES} per species) was manually reviewed and categorized by "
    f"dominant viewing angle. This sample indicated approximately {top_pct}% "
    f"top/cap views, {gill_pct}% underside/gill views, and {side_pct}% "
    f"side/stem or mixed views."
)


# bar chart of the distribution, saved alongside the audit csv

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.barplot(x=overall_pct.index, y=overall_pct.values)
plt.ylabel("% of sampled images")
plt.title(f"viewing-angle distribution (n={n_total_labeled} sampled images)")
plt.savefig(AUDIT_CSV.parent / "view_angle_distribution.png")
plt.show()

In [ ]:
# augmentation showcase, visualizing each transform step individually
# Standalone demo script

!pip install -q albumentations

import albumentations as A
import cv2
import numpy as np
import random
import zipfile
import time
from pathlib import Path
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

# same path
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/dataset_resplit.zip")
LOCAL_ZIP_PATH = Path("/content/dataset_resplit.zip")
LOCAL_DATA_ROOT = Path("/content/dataset_resplit")

if not LOCAL_ZIP_PATH.exists():
    print("Copying dataset_resplit.zip from Drive to local disk...")
    t0 = time.time()
    !cp "{DRIVE_ZIP_PATH}" "{LOCAL_ZIP_PATH}"
    print(f"Copy done in {time.time() - t0:.1f}s")
else:
    print("Local zip already present, skipping copy.")

if not LOCAL_DATA_ROOT.exists():
    print("Extracting dataset locally...")
    t0 = time.time()
    with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA_ROOT)
    print(f"Extraction done in {time.time() - t0:.1f}s")
else:
    print("Local dataset folder already present, skipping extraction.")

DATA_ROOT = LOCAL_DATA_ROOT / "dataset"
assert (DATA_ROOT / "Train").exists(), (
    f"Expected Train/ under {DATA_ROOT} but it's missing, check the zip structure with "
    f"'!unzip -l {LOCAL_ZIP_PATH} | head -30' and adjust DATA_ROOT manually."
)
print(f"DATA_ROOT confirmed: {DATA_ROOT}")

TRAIN_DIR = DATA_ROOT / "Train"

# three headline species
SPECIES = {
    "False Parasol\n(Chlorophyllum molybdites)": "Chlorophyllum_molybdites",
    "Stinking Dapperling\n(Lepiota cristata)": "Lepiota_cristata",
    "Yellow Stainer\n(Agaricus xanthodermus)": "Agaricus_xanthodermus",
}

IMG_SIZE = 384
demo_seed = 7  # seed used

# individual augmentation steps
individual_transforms = {
    "Original": None,  # placeholder, handled separately
    "RandomResizedCrop": A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0), ratio=(0.85, 1.15), p=1.0),
    "HorizontalFlip": A.HorizontalFlip(p=1.0),
    "VerticalFlip": A.VerticalFlip(p=1.0),
    "RandomRotate90": A.RandomRotate90(p=1.0),
    "GaussianBlur": A.GaussianBlur(blur_limit=(7, 7), p=1.0),  # fixed blur strength for a visible effect in the demo
    "ColorJitter": A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=1.0),
    "CoarseDropout": A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=1.0),
}

#  combined pipeline
full_pipeline = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0), ratio=(0.85, 1.15), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.15),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=0.5),
    A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
])


def find_sample_image(class_dir):
    """Returns the path of the first .jpg found in a class folder."""
    matches = list(class_dir.glob("*.jpg"))
    if not matches:
        raise FileNotFoundError(f"No .jpg images found in {class_dir}")
    return matches[0]


def load_rgb(path):
    image = cv2.imread(str(path))
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


step_names = list(individual_transforms.keys()) + ["Full Pipeline"]
n_rows = len(SPECIES)
n_cols = len(step_names)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.4 * n_rows))

for row_idx, (display_name, folder_name) in enumerate(SPECIES.items()):
    class_dir = TRAIN_DIR / folder_name
    img_path = find_sample_image(class_dir)
    raw_image = load_rgb(img_path)
    raw_resized = cv2.resize(raw_image, (IMG_SIZE, IMG_SIZE))

    for col_idx, step_name in enumerate(step_names):
        ax = axes[row_idx, col_idx]

        random.seed(demo_seed)
        np.random.seed(demo_seed)

        if step_name == "Original":
            display_img = raw_resized
        elif step_name == "Full Pipeline":
            display_img = full_pipeline(image=raw_image)["image"]
        else:
            transform = individual_transforms[step_name]
            display_img = transform(image=raw_image)["image"]
            if display_img.shape[:2] != (IMG_SIZE, IMG_SIZE):
                display_img = cv2.resize(display_img, (IMG_SIZE, IMG_SIZE))

        ax.imshow(display_img)
        ax.axis("off")
        if row_idx == 0:
            ax.set_title(step_name, fontsize=11, fontweight="bold")
        if col_idx == 0:
            ax.text(-0.15, 0.5, display_name, transform=ax.transAxes,
                     fontsize=10, va="center", ha="right", rotation=0)

plt.tight_layout()

OUTPUT_PATH = Path("/content/drive/MyDrive/augmentation_showcase.png")
plt.savefig(OUTPUT_PATH, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved augmentation showcase figure to: {OUTPUT_PATH}")

In [ ]:
# data resplitting, group-aware splitting.

!pip install -q scikit-learn

from pathlib import Path
from collections import defaultdict
import shutil
import zipfile
import time
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit

seed = 42

from google.colab import drive
drive.mount('/content/drive')

# paths used (fixed paths used by the researchers)
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/dataset.zip")          # original (current) dataset
LOCAL_ZIP_PATH = Path("/content/dataset.zip")
LOCAL_DATA_ROOT = Path("/content/dataset")                            # extraction target (original)
RESPLIT_ROOT = Path("/content/dataset_resplit_root")                  # new split, local staging
RESPLIT_ZIP_LOCAL = Path("/content/dataset_resplit.zip")
DRIVE_RESPLIT_ZIP_PATH = Path("/content/drive/MyDrive/dataset_resplit.zip")

if not LOCAL_ZIP_PATH.exists():
    print("Copying dataset.zip from Drive to local disk...")
    t0 = time.time()
    !cp "{DRIVE_ZIP_PATH}" "{LOCAL_ZIP_PATH}"
    print(f"Copy done in {time.time() - t0:.1f}s")
else:
    print("Local zip already present, skipping copy.")

if not LOCAL_DATA_ROOT.exists():
    print("Extracting dataset locally...")
    t0 = time.time()
    with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA_ROOT)
    print(f"Extraction done in {time.time() - t0:.1f}s")
else:
    print("Local dataset folder already present, skipping extraction.")

DATA_ROOT = LOCAL_DATA_ROOT / "dataset"
assert (DATA_ROOT / "Train").exists(), f"Expected Train/ under {DATA_ROOT}, check zip structure."
print(f"DATA_ROOT confirmed: {DATA_ROOT}")


print("\nCollecting all images and grouping by observation ID...")

all_paths = []
all_classes = []
all_group_ids = []

for split in ("Train", "Val", "Test"):
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        continue
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        for img_path in class_dir.glob("*.jpg"):
            group_id = img_path.stem.split("_")[0]
            all_paths.append(img_path)
            all_classes.append(class_dir.name)
            all_group_ids.append(group_id)

print(f"Total images: {len(all_paths)}")
print(f"Unique observation groups: {len(set(all_group_ids))}")

class_names_sorted = sorted(set(all_classes))
y = np.array([class_names_sorted.index(c) for c in all_classes])
groups = np.array(all_group_ids)

print("\nPerforming group-aware split (random_state fixed for reproducibility)...")

gss1 = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=seed)
train_idx, rest_idx = next(gss1.split(all_paths, y, groups))

rest_groups = groups[rest_idx]
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=seed)
val_rel_idx, test_rel_idx = next(gss2.split(rest_idx, y[rest_idx], rest_groups))
val_idx = rest_idx[val_rel_idx]
test_idx = rest_idx[test_rel_idx]

split_map = {"Train": train_idx, "Val": val_idx, "Test": test_idx}

print("\nResulting split sizes (may differ slightly from the original 80/10/10")
print("by IMAGE count, since whole observation groups move together")
print("group sizes vary, so exact per-split percentages aren't guaranteed,")
print("only group-level integrity is):")
for split_name, idxs in split_map.items():
    print(f"  {split_name}: {len(idxs)} images, {len(set(groups[idxs]))} groups")

print("\nRebuilding split with group-aware assignment...")

if RESPLIT_ROOT.exists():
    shutil.rmtree(RESPLIT_ROOT)

for split_name, idxs in split_map.items():
    for i in tqdm(idxs, desc=f"Copying {split_name}"):
        src = all_paths[i]
        cls = all_classes[i]
        dst_dir = RESPLIT_ROOT / "dataset" / split_name / cls
        dst_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst_dir / src.name)

print("Rebuild complete.")

print("\nZipping resplit dataset...")
if RESPLIT_ZIP_LOCAL.exists():
    RESPLIT_ZIP_LOCAL.unlink()
shutil.make_archive(str(RESPLIT_ZIP_LOCAL).replace(".zip", ""), 'zip', root_dir=str(RESPLIT_ROOT))
print(f"Zipping done in {time.time() - t0:.1f}s")

print("\nCopying zip to Drive...")
t0 = time.time()
!cp "{RESPLIT_ZIP_LOCAL}" "{DRIVE_RESPLIT_ZIP_PATH}"
print(f"Copy to Drive done in {time.time() - t0:.1f}s")
print(f"Saved: {DRIVE_RESPLIT_ZIP_PATH}")

print("\n" + "="*70)
print("VERIFYING NEW SPLIT - leakage check")
print("="*70)

def group_dataset_by_id(root_dir, splits=("Train", "Val", "Test")):
    groups_check = defaultdict(list)
    for split in splits:
        split_dir = root_dir / split
        if not split_dir.exists():
            continue
        for class_dir in sorted(split_dir.iterdir()):
            if not class_dir.is_dir():
                continue
            for img_path in class_dir.glob("*.jpg"):
                obs_id = img_path.stem.split("_")[0]
                groups_check[obs_id].append((split, class_dir.name, img_path.name))
    return groups_check

new_data_root = RESPLIT_ROOT / "dataset"
groups_check = group_dataset_by_id(new_data_root)
leaks = {h: v for h, v in groups_check.items() if len(v) > 1 and len({s for s, _, _ in v}) > 1}
total_leaked_images = sum(len(v) for v in leaks.values())

print(f"Hash groups with cross-split matches: {len(leaks)}")
print(f"Total images involved: {total_leaked_images}")
if leaks:
    print("Leakage still present - something went wrong, investigate before using this split.")
    for h, entries in list(leaks.items())[:10]:
        print(f"  {entries}")
else:
    print("Zero cross-split leakage confirmed. This dataset is clean.")

print(f"\nFinal split sizes on disk:")
for split in ("Train", "Val", "Test"):
    split_dir = new_data_root / split
    n_images = sum(1 for c in split_dir.iterdir() if c.is_dir() for _ in c.glob("*.jpg"))
    print(f"  {split}: {n_images} images")

print(f"\nDone. Clean, re-split dataset saved to Drive at: {DRIVE_RESPLIT_ZIP_PATH}")
print("Point DRIVE_ZIP_PATH at this file (dataset_resplit.zip) in future")
print("training scripts instead of the original dataset.zip.")

In [ ]:
# This is a normal py file meant to run localy.
# Copy the code if intended to use for testing.
import requests
import json
import time
import os
from pathlib import Path
import argparse
from urllib.parse import urlparse

# Species mapping: common folder name -> scientific name
SPECIES = {
    "Stinking_Dapperling": "Lepiota cristata",
    "Yellow_Stainers": "Agaricus xanthodermus",
    "False_Parasol": "Chlorophyllum molybdites",
    "Parasol": "Macrolepiota procera",
    "Field_Mushroom": "Agaricus campestris",
    "Shaggy_Parasol": "Chlorophyllum rhacodes",
}

def download_image(url, save_path):
    try:
        r = requests.get(url, stream=True, timeout=30)
        if r.status_code == 200:
            with open(save_path, 'wb') as f:
                for chunk in r.iter_content(1024):
                    f.write(chunk)
            return True
    except Exception as e:
        print(f"  Download failed: {e}")
    return False

def scrape_species(taxon_name, output_dir, max_images=8000):
    out_dir = Path(output_dir) / taxon_name.replace(" ", "_")
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nScraping {taxon_name} → {out_dir}")

    page = 1
    per_page = 100
    downloaded = 0
    base_url = "https://api.inaturalist.org/v1/observations"

    while downloaded < max_images:
        params = {
            "taxon_name": taxon_name,
            "quality_grade": "research",
            "has_photos": "true",
            "order_by": "observed_on",
            "order": "desc",
            "per_page": per_page,
            "page": page,
            "photo_license": "cc-by,cc-by-nc,cc0"
        }
        try:
            resp = requests.get(base_url, params=params, timeout=20)
            if resp.status_code != 200:
                print(f"  API error {resp.status_code}: {resp.text[:200]}")
                break
            data = resp.json()
            results = data.get("results", [])
            if not results:
                print("  No more results.")
                break
            for obs in results:
                if downloaded >= max_images:
                    break
                photos = obs.get("photos", [])
                for photo in photos:
                    if downloaded >= max_images:
                        break
                    url = photo.get("url")
                    if not url:
                        continue
                    large_url = url.replace("square", "large").replace("medium", "large").replace("small", "large")

                    filename = f"{obs['id']}_{photo['id']}.jpg"
                    save_path = out_dir / filename

                    if save_path.exists():
                        downloaded += 1
                        continue

                    if download_image(large_url, save_path):
                        downloaded += 1
                        if downloaded % 50 == 0:
                            print(f"  Downloaded {downloaded}/{max_images}")
            page += 1
            time.sleep(1.0)
        except Exception as e:
            print(f"  Error on page {page}: {e}")
            time.sleep(5)
            break

    print(f"Finished {taxon_name}: {downloaded} images")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--output", default=r"F:\Downloads\Mushroom Datasets\inaturalist_scraped", help="Output root folder")
    parser.add_argument("--max_per_species", type=int, default=8000)
    # --max_per_species a count of 10000 is used for the dataset
    args = parser.parse_args()

    root = Path(args.output)
    root.mkdir(parents=True, exist_ok=True)

    print("Starting iNaturalist scraper for 6 target mushrooms...")

    for folder_name, taxon in SPECIES.items():
        scrape_species(taxon, root, args.max_per_species)

    print("\nAll done! Images organized by species folders.")

if __name__ == "__main__":
    main()